# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv


In [2]:
import dask.dataframe as dd
from glob import glob
import os
import pandas as pd

c:\Users\myche\.conda\envs\dsi_participant\lib\site-packages\dask\dataframe\_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
# Load the environment variable, PRICE_DATA
# the information for this environment variable is stored in the .env file
price_file = os.getenv("PRICE_DATA")

# Use glob to find the path of all parquet files in the directory PRICE_DATA
price_glob = glob(price_file + '/**/*.parquet', recursive=True)
price_df = dd.read_parquet(price_glob)

# Remove the "Price" as the name of the first column
price_df.columns.name = None
price_df

,Date,Adj Close,Close,High,Low,Open,Volume,Year
npartitions=13078,,,,,,,,
,"datetime64[ns, UTC]",float64,float64,float64,float64,float64,float64,int32
,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [4]:
print(price_df.columns)
#print(price_df.head())
print(price_df.index)

Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Year'], dtype='object')
<dask_expr.expr.Index: expr=Index(frame=ReadParquetFSSpec(c937a0d))>


In [5]:
# Convert the Ticker from index into a regular column
price_df = price_df.reset_index()
price_df["Ticker"]

# Define a function to apply shift within each partition
def add_lags(price_df):
    price_df = price_df.sort_values(by=["Ticker","Date"])
    price_df["Close_lag_1"] = price_df["Close"].shift(1)
    price_df["Adj_Close_lag_1"] = price_df["Adj Close"].shift(1)
    price_df["returns"] = (price_df["Close"] / price_df["Close_lag_1"]) - 1
    price_df["hi_lo_range"] = price_df["High"] - price_df["Low"]
    return price_df

# # Group by Ticker and apply the function
dd_feat = price_df.groupby("Ticker").apply(add_lags)
dd_feat = dd_feat.persist()
dd_feat

C:\Users\myche\AppData\Local\Temp\ipykernel_27380\1366544986.py:15: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = price_df.groupby("Ticker").apply(add_lags)


,Ticker,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
npartitions=13078,,,,,,,,,,,,,
,object,"datetime64[ns, UTC]",float64,float64,float64,float64,float64,float64,int32,float64,float64,float64,float64
,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [6]:
# Convert the Dask dataframe to a pandas data frame
dd_feat_pds = dd_feat.compute()

# Add a new feature that contains the average of returns using a window of 10 days
dd_feat_pds["Rolling Average"] = dd_feat_pds["returns"].rolling(10).mean()

In [7]:
dd_feat_pds

Price     Ticker                      Date  Adj Close      Close       High  \
Ticker                                                                        
DOV    0     DOV 2000-01-03 00:00:00+00:00  18.544512  29.410976  30.123207   
       1     DOV 2000-01-04 00:00:00+00:00  18.201101  28.866327  29.494768   
       2     DOV 2000-01-05 00:00:00+00:00  18.280350  28.992016  29.285288   
       3     DOV 2000-01-06 00:00:00+00:00  18.359598  29.117704  29.662352   
       4     DOV 2000-01-07 00:00:00+00:00  18.861513  29.913729  29.997520   
...          ...                       ...        ...        ...        ...   
CTLT   15   CTLT 2025-01-27 00:00:00+00:00        NaN        NaN        NaN   
       16   CTLT 2025-01-28 00:00:00+00:00        NaN        NaN        NaN   
       17   CTLT 2025-01-29 00:00:00+00:00        NaN        NaN        NaN   
       18   CTLT 2025-01-30 00:00:00+00:00        NaN        NaN        NaN   
       19   CTLT 2025-01-31 00:00:00+00:00        NaN        NaN        NaN   

Price            Low       Open     Volume  Year  Close_lag_1  \
Ticker                                                          
DOV    0   29.243393  29.997520   730380.0  2000          NaN   
       1   28.782536  29.410976   404275.0  2000    29.410976   
       2   28.782536  28.782536   930877.0  2000    28.866327   
       3   28.824432  28.824432   993532.0  2000    28.992016   
       4   29.033913  29.159599  1742262.0  2000    29.117704   
...              ...        ...        ...   ...          ...   
CTLT   15        NaN        NaN        NaN  2025          NaN   
       16        NaN        NaN        NaN  2025          NaN   
       17        NaN        NaN        NaN  2025          NaN   
       18        NaN        NaN        NaN  2025          NaN   
       19        NaN        NaN        NaN  2025          NaN   

Price      Adj_Close_lag_1   returns  hi_lo_range  Rolling Average  
Ticker                                                              
DOV    0               NaN       NaN     0.879814              NaN  
       1         18.544512 -0.018519     0.712233              NaN  
       2         18.201101  0.004354     0.502752              NaN  
       3         18.280350  0.004335     0.837919              NaN  
       4         18.359598  0.027338     0.963608              NaN  
...                    ...       ...          ...              ...  
CTLT   15              NaN       NaN          NaN              NaN  
       16              NaN       NaN          NaN              NaN  
       17              NaN       NaN          NaN              NaN  
       18              NaN       NaN          NaN              NaN  
       19              NaN       NaN          NaN              NaN  

[3173427 rows x 14 columns]

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

It was not neccessary to convert to Pands to calculate the moving average. This is possible to do in dask using operations such as .rolling() without having to do the conversion. It would have been better to do this in Dask because it is more efficient in processing larger data without running into memory limitations.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.